![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Lab: Production-ready RAG with caching and memory
---

## Introduction

Let’s get started with the second lab of the RAG chatbot course. Make sure you have completed the first lab (`01_redisvl_basic_rag`) before starting this one. 

In the previous lab, you built a complete RAG pipeline from scratch. You loaded and embedded a real-world document (Nike’s 10-K financial filing), indexed it in Redis, and used OpenAI to generate answers grounded in semantic search results.

That’s a solid foundation - but before you can ship this system to production, you need to address two critical limitations:

- Cost and latency: Every user query results in a call to the LLM — even for repeat or nearly identical questions.

- Memory: Our chat has no memory of prior exchanges. Each question is treated in isolation, breaking conversational continuity.

In other words, your current pipeline works, but it’s not yet optimized for real users or traffic.

This lab introduces two essential upgrades:

- LLM Caching: Save and reuse past responses by checking Redis for semantically similar queries.

- Session Memory: Maintain chat history across multiple user questions to give the LLM a running sense of context.

This lab is broken out into the following sections:

1. Introduction (📍 You are here)  
2. Add LLM caching  
3. Add session memory  
4. Run the full chatbot  
5. Wrap up

By the end, you’ll have a more intelligent, cost-efficient, and production-ready RAG application — powered end-to-end by Redis.

Additionally, if you ever want to view the database at any point during the lab, you can return to the lab documentation page and open Redis Insight.

Let’s get started!

## Set up the notebook environment  

This lab's material picks up where the previous lab ended. The `setup-lab-prod.py` script (located in the `/resources-rag` directory) will reproduce the notebook's results here. Execute the code block below to run the script.

⚠️ **Note:** The setup file may take up to two minutes to complete. You don't need to wait for it to complete before you continue. Any additional code execution will be queued.

In [ ]:
# Run the setup script for Lab 3
%run ./resources-rag/setup-lab-prod.py

## Adding semantic caching to improve RAG performance

First, we'll focus on reducing redundant LLM calls to the app using semantic caching.

With a semantic cache, before we make a new LLM request, the system will embed the input and run a search in the cache. If there’s a “close” match (as defined by the cache settings), the system returns the cached answer and skips the LLM call entirely. That means zero LLM cost, near-instant response time, and more efficient operation.

### RedisVL SemanticCache

RedisVL includes a class called [`SemanticCache`](https://docs.redisvl.com/en/stable/api/cache.html) for easy LLM caching. 

Examine the cache defined in the code block below and then run it.

In [ ]:
from redisvl.extensions.cache.llm import SemanticCache

llmcache = SemanticCache(
    name="llmcache",                
    vectorizer=hf,                  
    redis_url=REDIS_URL,          
    ttl=300,                       
    distance_threshold=0.2,        
    overwrite=True      
)

print("Cache created successfully:", llmcache)

<details>
<summary> (Optional) Semantic cache parameter breakdown </summary>

We have included the following properties for the cache: 

- `name`: An optional argument that sets the name of the semantic cache search index. Defaults to `“llmcache”`.

- `vectorizer`: An optional argument that sets the vectorizer for the cache.

- `ttl`: An optional argument that sets the "time to live" in seconds for a cached result. After 2 minutes, entries will expire.

- `distance_threshold`: The maximum distance for considering a cache hit. This defaults to 0.1.

- `overwrite`: Whether or not to force overwrite the schema for the semantic cache index. Defaults to false.

</details>

When you ran the code above, RedisVL also created a new index (named "llmcache"). 

📌 **Task 1: Examine the index with RedisVL CLI**  

Use the [RedisVL CLI](https://docs.redisvl.com/en/latest/overview/cli.html) to inspect the index that was created.


In [ ]:
# Put your RedisVL CLI command below to inspect the index


# ---

<details>
<summary> 🔒 RedisVL CLI Solution </summary>
<br>

```

!rvl index info -i llmcache
```

</details>

With the cache defined, we can now implement a way for it to apply to our LLM calls.

### Implementing the cache in the RAG app

There are a few different ways that the cache can be built into the existing app from the previous lab. We're going to keep it simple and create a new function called `answer_question_with_cache()` and use our previously defined `answer_question()` function inside of it. 

As a reminder, this is what our `answer_question` function looks like: 

```python
async def answer_question(index: AsyncSearchIndex, query: str):
    """End-to-end RAG: embeds query, retrieves context, generates LLM response"""

    # Step 1: Embed the query
    query_vector = embed_query(query)

    # Step 2: Retrieve matching context chunks from Redis
    context = await retrieve_context(index, query_vector)

    # Step 3: Generate response from OpenAI using system prompt + context
    return await generate_llm_response(query, context)
```

Our new cache function will work largely the same, except that before it calls `.answer_question()` it will check the cache for an existing answer. If it exists, it will simply return the cached response. If no response is cached that fits, it will perform the retrieval and then store the response in the cache for the future.

Examine the new function's implementation below and then run the code block. The code block will run silently.

In [ ]:
import time

async def answer_question_with_cache(index, query_text):

    # Grab the start time for performance measurement
    start_time = time.time()

    # Embed the query text
    query_vector = embed_query(query_text)

    # Check cache
    result = llmcache.check(vector=query_vector)

    # If cache hit: set the response and return
    if result:
        elapsed = time.time() - start_time
        return {
            "text": result[0]['response'],
            "cached": True,
            "elapsed": elapsed
        }

    # If cache miss: go through full RAG pipeline
    response = await answer_question(index, query_text)
    llmcache.store(query_text, response, query_vector)

    # Calculate elapsed time and return the response
    elapsed = time.time() - start_time
    return {
        "text": response,
        "cached": False,
        "elapsed": elapsed
    }

Notice that we used two methods with our cache:

- `.check()`: This method checks the semantic cache for results similar to the specified prompt or vector. If a close enough match is found (based on the distance threshold), it returns the cached response; otherwise, it returns an empty list (`[]`).

- `.store()`: This method stores the specified key-value pair in the cache along with metadata. In our case, we are storing the query, its response, and the associated embedding vector.

We also changed up the implementation to visualize the speed at which our response will return. This will help you see how much faster the program runs with a cache. Feel free to check out how this works with the UI interface in the `chat_ui.py` file (around line 99) inside of the `resources-rag` directory.


## Test the cache out

With the cache implemented, we can now test the functionality. Below is a code block that will boot up the UI chat interface from the previous lab. Run the code block to create the chat interface and then try asking the two queries below to see the cache in action.

1. What was Nike's revenue last year compared to this year?
2. What was Nike's total revenue in the last year compared to now?

In [ ]:
import sys
sys.path.append('resources-rag')
from chat_interface import start_chat

start_chat(answer_question_with_cache, async_index)

## Adding memory

With caching implemented, we can tackle the issue of memory. Our chat still answers each question independently — with no awareness of what came before (other than if the query is "similar" to one in the cache). 

To make our app more conversational and personalized, we'll want to preserve chat history. This will allow our chat app to understand follow-up questions or references to earlier answers.

In this section, we’ll implement storing the chat history using RedisVL's [`SemanticMessageHistory`](https://docs.redisvl.com/en/latest/api/message_history.html) class. 


### RedisVL SemanticMessageHistory

Let's import the class responsible for helping with session history: `SemanticMessageHistory`. 

Run the code block below.

In [ ]:
from redisvl.extensions.message_history import SemanticMessageHistory

### A reusable assistant

At this point, we can start treating our RAG logic as a reusable assistant — so we'll wrap it in a class. This gives us a clean way to manage state (like session memory), reuse components (like the vectorizer and index), and organize our logic for answering questions more effectively.

Once the class is complete, we'll be able to call it like so: 

```python
chat = ChatBot(async_index, vectorizer=hf, user="Tom")
```

We'll also build the ability to run two primary methods on the class instance:

1. `.answer_question()`: We'll use this to run the full RAG flow with caching and session memory. We'll upgrade the logic of our existing caching version of this function from earlier.

2. `.clear_history()`: We'll use this to reset the conversation memory for a clean slate

To start, let's create the class constructor. Using the structure above, it will need to take in an index, a vectorizer object, and a user name. We'll also define the session manager as a property of our class. 

Examine and then run the code block below.

In [ ]:
class ChatBot:
    def __init__(self, index, vectorizer, user: str):
        self.index = index
        self.vectorizer = vectorizer
        self.session_manager = SemanticMessageHistory(
            name=f"chat_session_{user}",
            redis_url=REDIS_URL, # redis_url was imported from the setup script
        )

The important argument to call out from above is `name`. This property defines the name of the Redis index where messages will be stored. In this example, we include the user's name in the index name (e.g., chat_session_tom) to give each user their own dedicated index.
 
An additional property to be aware of that we didn't assign above is the `session_tag` property. Setting this property can distinguish individual sessions. When it is not filled, it will default to a instance ULID, but it would be common to want to use the tag to describe what each different user session is about (e.g., tesla_stock_analysis, meta_ethics).


### Attaching existing methods to the new class

Next, we'll attach the existing methods we built in the previous lab to the class. This includes slightly modified versions (to fit the class structure) of `embed_query()`, `retrieve_content()`, and `generate_llm_response()`. Visit the `setup-lab-prod.py` file to check them out. 

Run the code block below to attach them to the class.

In [ ]:
ChatBot.embed_query = embed_query_class
ChatBot.retrieve_context = retrieve_context_class
ChatBot.generate_llm_response = staticmethod(generate_llm_response)

### Clear chat history

As mentioned earlier, one piece of functionality we'll also want as part of our class is the ability to clear the session history. This will enable us to start new sessions with a clean slate. This can be implemented by using the provided `.clear()` method from the `SemanticMessageHistory` class.

📌 **Task 2: Clear history method**  

Create a new function in the code block below called `clear_history`. The method should call `.clear()` on the session manager object we created earlier. Then, on a separate line, attach the method to be part of the `ChatBot` class.

The solution is posted below. Make sure to confirm your solution and then run the code block.

In [ ]:
# --- Write the clear_history function below ---



<details>
<summary> 🔒 Clear History Solution Code </summary>
<br>

```python
async def clear_history(self):
    """Clear session chat"""
    self.session_manager.clear()

ChatBot.clear_history = clear_history
```

</details>

## Query functionality

Lastly, but most importantly, we'll need to build the class method (defined as a function and then attached) for answering the query passed to the chat. Examine the following incomplete code block:

In [ ]:
async def answer_question_with_cache_and_history(self, query: str):
    """Answer the user's question with session context and caching baked-in"""

    # Start timer to track performance
    start_time = time.time()

    # Initialize flags for cache and memory usage
    used_cache = False
    used_memory = False

    # 📌 Task A1: Embed the query using class helper method
    

    # 📌 Task A2: Check the semantic cache


    # If cache hit, retrieve the answer and set the used_cache flag
    if result:

        answer = result[0]['response']
        used_cache = True

    else:
        # Retrieve current session messages
        session = self.session_manager.messages
        used_memory = bool(session)


        # 📌 Task A3: Retrieve context using class helper

    
        # 📌 Task A4: Generate answer using LLM


        # 📌 Task A5: Store response in semantic cache
        
        
    # Update session memory and return the answer
    self.session_manager.add_messages([
        {"role": "user", "content": query},
        {"role": "assistant", "content": answer}
    ])

    # Return answer + metadata
    return {
        "text": answer,
        "cached": used_cache,
        "used_memory": used_memory,
        "elapsed": time.time() - start_time
    }

# Attaching the new method to the ChatBot class
ChatBot.answer_question_with_cache_and_history = answer_question_with_cache_and_history

You've probably noticed that the `.answer_question_with_cache_and_history()` method is missing a few key pieces. We've already included the session manager logic for handling chat history, but you’ll need to complete the rest of the method using the helper functions we attached to the `ChatBot` class. 

Go through each of the tasks below and complete the method.

Feel free to reference `setup-lab-prod.py` if you want a refresh on each of the methods we are using from the previous lab. Additionally, after completing each task, confirm your answer by reviewing the solution attached to each task. 


<details>

<summary> 📌 <b> Task A1: Embed the query </b></summary>
<br>

First, just like before, we need to turn the user’s question into a vector so we can compare it by meaning. We already attached an `embed_query()` helper to the class — now we just need to call it (with any associated arguments) inside the method.

Assign the result to a variable called `query_vector`.

<details>
<summary> 🔒 Solution</summary>
<br>

```python
query_vector = self.embed_query(query)
```
</details>

</details>

<details>
<summary> 📌 <b> Task A2: Check the cache for similar queries </b></summary> 

We want to make sure to check the cache for similar queries. Use the appropriate method from the `llmcache` class to check for existing results. If you find a match, assign the answer to a variable called `answer`. Note that we have filled in `None` for this section as a placeholder. This will need to be replaced with logic that checks whether a similar query has already been answered

<details>
<summary> 🔒 Hint + Solution</summary>
<br>

**Hint:** Recall the method for checking if a cache has a particular value is `.check()`.

<details>
<summary> 🔒 Solution</summary>
<br>

```python
result = llmcache.check(vector=query_vector)
```
</details>

</details>

</details>

<details>
<summary> 🔍 <b> Review Code: Retrieve current session messages </b> </summary>

Notice we have the following code:

```python
session = self.session_manager.messages
used_memory = bool(session)
```

This code gets the current conversation history (session_manager.messages) and sets `used_memory` to `True` if any messages exist. If there's no cache hit, that history helps the LLM answer with context.

You don’t need to change this line, but remember you can retrieve the current conversation history by calling the `messages` property. 

</details>


<details>
<summary> 📌 <b> Task A3: Retrieve context </b></summary>
<br>

If there wasn’t a cache hit, we’ll need to pull relevant documents from Redis to serve as context for our LLM call. We already attached an `.retrieve_context()` helper to the class  — now we just need to call it (with any associated arguments) inside the method.

Assign the result to a variable called `context`.

<details>
<summary> 🔒 Solution</summary>
<br>

```python
context = await self.retrieve_context(query_vector)
```
</details>

</details>

<details>

<summary> 📌 <b> Task A4: Generate answer </b></summary>
<br>

Now that you have the query, the retrieved context, and the session history, it's time to generate a response from the LLM.

Before writing this code, examine the first part of the new version of our `generate_llm_response()` from the `setup-lab-prod.py` file:

```python
async def generate_llm_response(query: str, context: str, session: list) -> str:
    """Construct and send a complete prompt to the OpenAI LLM and return its response."""

    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}] +
        session +
        [{"role": "user", "content": promptify(query, context)}]
    )
```

Notice that it now takes a `session` argument to pass along with the messages. This is because we want our LLM call to consider the context of the session history when creating a reply.

We already attached a `generate_llm_response()` helper to the class — now we just need to call it (with any associated arguments) inside the method.

Assign the result to a variable called `answer`.

<details>
<summary> 🔒 Solution</summary>
<br>

```python
answer = await self.generate_llm_response(query, context, session)
```
</details>

</details>

<details>

<summary> 📌 <b>Task A5: Store the response in the cache </b></summary>

<br>

Next, we'll want to store the response from the LLM call to the cache. Use the appropriate method from the `llmcache` class to store the result from the earlier step. Make sure to pass the method the following:

1. The original query (`query`)
2. The answer from the LLM call (`answer`)
3. The query vector created (`query_vector`)

<details>
<summary> 🔒 Hint + Solution</summary>
<br>

**Hint:** Recall the method for storing a particular value in the cache is `.store()`.

<details>

<br>

<summary> 🔒 Solution</summary>
<br>

```python
llmcache.store(query, answer, query_vector)
```
</details>

</details>

</details>

<details>
<summary> 🔍 <b> Review Code: Update session memory and return the answer </b> </summary>

Examine the following code: 

```python
self.session_manager.add_messages([
        {"role": "user", "content": query},
        {"role": "assistant", "content": answer}
    ])
```

This code updates the conversation history by adding the latest interaction — both the user's question and the assistant’s response — using the `.add_messages()` method. Like before, each message is a dictionary with a role (like "user" or "assistant") and some content.

You don’t need to change this line, but keep in mind that session memory only works effectively if it’s updated whether the answer is cached or not. This is done through the `.add_messages()` method.
</details>


<details>
<summary> 🔒 <b>Full Solution Code </b> </summary>

```python

async def answer_question_with_cache_and_history(self, query: str):
    """Answer the user's question with session context and caching baked-in"""

    # Start timer to track performance
    start_time = time.time()

    # Initialize flags for cache and memory usage
    used_cache = False
    used_memory = False


    # 📌 Task A1: Embed the query using class helper method
    query_vector = self.embed_query(query)

    # 📌 Task A2: Check the semantic cache
    result = llmcache.check(vector=query_vector)

    # If cache hit, retrieve the answer and set the used_cache flag
    if result:

        answer = result[0]['response']
        used_cache = True

    else:
        # Retrieve current session messages
        session = self.session_manager.messages
        used_memory = bool(session)


        # 📌 Task A3: Retrieve context using class helper
        context = await self.retrieve_context(query_vector)
    
        # 📌 Task A4: Generate answer using LLM
        answer = await self.generate_llm_response(query, context, session)

        # 📌 Task A5: Store response in semantic cache
        llmcache.store(query, answer, query_vector)
        

    # Update session memory and return the answer
    self.session_manager.add_messages([
        {"role": "user", "content": query},
        {"role": "assistant", "content": answer}
    ])

    # Return answer + metadata
    return {
        "text": answer,
        "cached": used_cache,
        "used_memory": used_memory,
        "elapsed": time.time() - start_time
    }

# Attaching the new method to the ChatBot class
ChatBot.answer_question_with_cache_and_history = answer_question_with_cache_and_history

```

</details>

## Test the entire RAG workflow

At this point, you’ve built a personalized RAG assistant with vector search, semantic caching, and session memory — all backed by Redis. You’ve modularized the logic into a reusable class, and you're now ready to interact with it live.

Similar to the previous lab, the code in this section will launch a chat UI. Before trying it out, start by creating a new chat session and clearing any previous history. Run the code block below.

In [ ]:
# Set up session
# Feel free to change the user name to your own
chat = ChatBot(async_index, vectorizer=hf, user="Tom")
await llmcache.aclear()
await chat.clear_history()

Now, you can launch the chat. Run the code block to boot it up. You can type your own questions about Nike’s 10-K filing or use the provided queries in the dropdowns to test caching and memory functionality.

<details>
<summary>Want to test semantic caching again? 🔁 </summary>
<br>

These two questions are phrased differently but ask for the same information. The second call should result in a cache hit, making the response nearly instant.

- What are Nike’s main operating segments?
- Can you describe Nike’s primary operating divisions and regions?
</details>

<details>
<summary> Want to test session memory? 🧠 </summary>
<br>

Try any of the following sequences of related questions:

A. 

1. What was Nike's marketing strategy?
2. What role do athlete partnerships play in that?


B. 

1. What are Nike’s main revenue drivers according to its 2023 10-K?
2. Tell me more about the direct sales.

</details>


In [ ]:
import sys
sys.path.append('resources-rag')
from chat_interface import start_chat_advanced

start_chat_advanced(chat)

## Wrap Up 🏁 

Great job! You've completed the lab and have built a production-ready RAG with caching and memory.

You took your RAG skills to the next level by implementing features that make real-world assistants feel faster, smarter, and more natural to use. Redis isn't just storing your data — it's powering the backbone of efficient and stateful LLM interactions.

More specifically, you learned how to:

- Use semantic caching to avoid redundant LLM calls and speed up response times
- Add session memory to preserve conversation history across turns

To continue learning, check out any of these additional resources:
- [Redis University](https://university.redis.io/): Become a Redis expert on our university platform
- [Redis AI Resources](https://github.com/redis-developer/redis-ai-resources): A curated repository of resources for basic and advanced Redis use cases in the AI ecosystem. 
- [Redis for AI Documentation](https://redis.io/docs/latest/develop/ai/): An overview of Redis for AI and search documentation
- [RedisVL](https://github.com/redis/redis-vl-python): The RedisVL GitHub Repo
- [LangCache](https://redis.io/langcache/): A managed semantic cache offering by Redis

To complete the course, and earn your certificate, return to the [Redis University course](https://flockjay.com/course/2qa1u1ss21vsy5/submodule/zshfn4ozytjpgr/) and complete the knowledge check. 